<a href="https://colab.research.google.com/github/Superfawll/ml_performance_playground/blob/main/jax_transformers_playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook we will work with Jax, mainly exploring some of its fundamental functionalities, its main philosophy, and how to write high performance / composite code using jax / pallas.

In [1]:
import jax
from jax import numpy as jnp                # the numpy  dual in jax
from jax._src.api import block_until_ready  # necessary for time measurement and instrumentation

jax.devices()                               # tells us how many of what devices jax can detect
jax.__version__

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


'0.7.2'

Generally, there are a couple of main semantic considerations in jax:


1.   Jax is based on array-like computations.
2.   Jax uses composition (grad, vmap, jvp/vjp, etc.) and pure functions (functions that only depend on their inputs) to compose more complex functionalities, and trace them compile-time to perform higher level optimizations (jit)
3. Jax's tracing symbolically executes a program on compile time, and records operations, shapes of the variables and their relationships. Later, using lowering rules, it converts them using one or more mapping primitives to StableHLO, which is a lower level representation than jax. XLA later uses those representations to perform fusions, scheduling and optimizations for the specific hardware it will run on.
4. Jax has auto differentiation. This means you can use several different functions, such as jax.grad (using VJP calculates the gradient of a function wrt its inputs),  jax.vjp / jax.jvp (VJP and JVP calculations in jax) jax.jacfwd / jax.jacrev (forward and reverse mode jacobian calculation) and jax.linearize (calculates the jvp and the value at a specific point.)

Particularly in this notebook we will work with a simple decoder transformer:

**input tokens -> embeddings -> Q, K, V and attention dot product -> MLP layer -> decode to vocabulary -> loss -> backprop**

and we use jax's functionalities to see how we can work some fun exemplary use-cases out.

Let's start by initializing some matrices for our transformers, and some dimensions:

In [2]:
# Initializing the parameters of our decoder
L, B, T, S, D, H, F, V = 6, 5, 5, 5, 8, 4, 10, 1024

key = jax.random.key(0)
key_embed, key_x, key_o, key_up, key_down, key_dec = jax.random.split(key, 6)

embedding_table = jax.random.normal(key_embed, (V, D), dtype=jnp.bfloat16)
tokens = jax.random.randint(key_x, (B, T + 1), minval=0, maxval=V)

l_rate = 0.001

W_Q = jnp.ones((D, H), dtype=jnp.bfloat16)
W_K = jnp.ones((D, H), dtype=jnp.bfloat16)
W_V = jnp.ones((D, H), dtype=jnp.bfloat16)

W_O = jax.random.uniform(key_o, (H, D), dtype=jnp.bfloat16)

W_up = jax.random.normal(key_up, (D, F), dtype=jnp.bfloat16)
W_down = jax.random.normal(key_down, (F, D), dtype=jnp.bfloat16)

W_dec = jax.random.normal(key_dec, (D, V), dtype=jnp.bfloat16)

weights = (W_Q, W_K, W_V, W_O, W_up, W_down, W_dec, embedding_table)

Let's start with a simple qkv projection and the attention dot-product:

In [3]:
def project_qkv(x, w_q, w_k, w_v):
    q = x @ w_q
    k = x @ w_k
    v = x @ w_v
    return q, k, v

def attention_dot_prod(q, k, v, w_o):
  B, T, S, H = q.shape[0], q.shape[1], k.shape[1], q.shape[2]

  attention_score = jnp.einsum('bth, bsh -> bts', q, k) / jnp.sqrt(H) # Q . KT
  mask_kernel = jnp.tril(jnp.ones((T, S), dtype=bool)) # causal mask

  masked_scores = jnp.where(mask_kernel, attention_score, -jnp.inf)
  normalized = jax.nn.softmax(masked_scores, axis=2) # softmax(normalized_scores[B, T, S])

  value_projection = normalized @ v # S . V -> [B, T, H]
  return value_projection @ w_o

As you can see, we have performed the attention dot-product using a very handy and interesting numpy function called einsum. This function uses einstein's notation to perform many different forms of matrix operations, like transpositions, complex matmults and so on.

At compile time, this function is a bit slower than the standard '@' or dot_general due to the parsing of the string notation, however, at runtime they will likely all map to the same set of backend implementations.

You can also see our causal masking, performed using a masking kernel we crafted with `jnp.tril`. We will see later that StableHLO and jaxpr implement this tril functionality using two `iota`s and conditioning.

Note also that our softmax for the attention dot-product is happening over the axis 2 of the normalized scores, which means each other token in the sequence receives a probability score.

We will then continue with MLP forward computation, the normalization and the cross-entropy loss calculator:

In [4]:

def mlp_forward(x, win, wout):
  return jax.nn.tanh(x @ win) @ wout

def layer_norm(x, epsilon):
  mean = jnp.mean(x, axis=-1, keepdims=True)
  variance = jnp.var(x, axis=-1, keepdims=True)

  return (x - mean) / jnp.sqrt(variance + epsilon)

def cross_ent_loss(x, x_target):
  B, T = x.shape[0], x.shape[1]

  log_probabilities = jax.nn.log_softmax(x, axis=2)

  batch_indices = jnp.arange(B)[:, None]
  token_indices = jnp.arange(T)[None, :]

  target_log_loss = -log_probabilities[batch_indices, token_indices, x_target]
  mean_target_log_loss = jnp.mean(target_log_loss)

  return mean_target_log_loss

Interestingly, even though we can calculate a mean, a variance and a normalized layer value with only three methods and depend on broadcasting rules to handle the rest, after compilation in jaxpr, StableHLO and XLA these are explicit broadcasts that need to happen to make the arrays match each other's shape and form.

The lower in the stack, the more of these explicit mappings, until in machine code or lower ASM you need to make sure all the shapes that regard the same computation match in any form they need to. Nothing will be implicit then unless its unnecessary.

We now group all of these functions above in our **pure** decoder layer function. For later use, we also wrap the portion of the code that calculates the projections, the dot-product, and the decode into the vocabulary in a separate function called "forward_logits":

In [5]:
def decoder_layer(tokens, weights):
  W_Q, W_K, W_V, W_O, W_up, W_down, W_dec, embedding_table = weights

  B, T = tokens.shape[0], tokens.shape[1] - 1

  target_tokens = tokens[:, 1: T + 1]
  input_tokens = tokens[:, :T]
  input_embeddings = embedding_table[input_tokens]

  output = forward_logits(input_embeddings, (W_Q, W_K, W_V, W_O, W_up, W_down, W_dec))

  return cross_ent_loss(output, target_tokens)

def forward_logits(input_embeddings, weights):
  W_Q, W_K, W_V, W_O, W_up, W_down, W_dec = weights

  q, k, v = project_qkv(input_embeddings, W_Q, W_K, W_V)

  v_proj = attention_dot_prod(q, k, v, W_O)

  normalized_attention = layer_norm(v_proj + input_embeddings, 0.00001)

  prediction = mlp_forward(normalized_attention, W_up, W_down) # BTD @ DF -> BTF, tanh(BTF) @ FD -> BTD
  output = layer_norm(prediction + normalized_attention, 0.00001) @ W_dec # BTD @ DV -> BTV

  return output

You can see here that we're using the "next-token-loss" framework, where we shift the input tokens by one point to serve as the targets for calculating the loss. This way the model learns to guess the next token at each position.

Let's look at the grouping of our decoder layer and backwards gradient calculation in `training_step`:

In [6]:
def training_step(tokens, weights, l_rate):
  loss_and_grad_fn = jax.value_and_grad(decoder_layer, argnums=1)
  loss, grads = loss_and_grad_fn(tokens, weights)

  new_weights = tuple(
    mat - l_rate * grad
    for mat, grad in zip(weights, grads)
  )

  weights = new_weights

  return weights, loss

def run_epochs(tokens, weights, l_rate, epoch_count = 1):
  new_weights = weights
  loss = 0
  for _ in range(epoch_count):
    new_weights, loss = training_step(tokens, new_weights, l_rate)

  return new_weights, loss

In [7]:
loss_1 = decoder_layer(tokens, weights)
updated_weights, loss = run_epochs(tokens, weights, l_rate, 1000)
loss_2 = decoder_layer(tokens, updated_weights)

loss_1, loss, loss_2

(Array(11.625, dtype=bfloat16),
 Array(11.5, dtype=bfloat16),
 Array(11.5, dtype=bfloat16))

In [8]:
compiled_training_step = jax.jit(training_step)

weights_plain, loss_plain = training_step(tokens, weights, l_rate)
weights_jit, loss_jit = compiled_training_step(tokens, weights, l_rate)

print(loss_plain, loss_jit)

%timeit -n 10 -r 5 jax.block_until_ready(compiled_training_step(tokens, weights, 0.001))
%timeit -n 10 -r 5 jax.block_until_ready(training_step(tokens, weights, 0.001))

11.625 11.625
332 µs ± 23.2 µs per loop (mean ± std. dev. of 5 runs, 10 loops each)
44.5 ms ± 667 µs per loop (mean ± std. dev. of 5 runs, 10 loops each)


In [9]:
print("=============================== AFTER   JIT ===============================")
print(compiled_training_step.trace(tokens, weights, l_rate).jaxpr)
print("====================== AFTER  LOWERING  TO STABLEHLO ======================")
print(compiled_training_step.trace(tokens, weights, l_rate).lower().as_text())
print("============================== AFTER COMPILE ==============================")
print(compiled_training_step.trace(tokens, weights, l_rate).lower().compile().as_text())

=============================== AFTER   JIT ===============================
let _var = { lambda ; a:bf16[5,5,8] b:i32[]. let
    c:f32[5,5,8] = convert_element_type[new_dtype=float32 weak_type=False] a
    d:f32[5,5] = reduce_sum[axes=(2,)] c
    e:f32[5,5,1] = broadcast_in_dim[
      broadcast_dimensions=(0, 1)
      shape=(5, 5, 1)
      sharding=None
    ] d
    f:f32[5,5,1] = div e 8.0:f32[]
    g:f32[5,5,8] = sub c f
    h:f32[5,5,8] = square g
    i:f32[5,5,8] = mul 2.0:f32[] g
    j:f32[] = convert_element_type[new_dtype=float32 weak_type=False] b
    k:f32[] = sub 8.0:f32[] j
    l:f32[5,5] = reduce_sum[axes=(2,)] h
    m:f32[5,5,1] = broadcast_in_dim[
      broadcast_dimensions=(0, 1)
      shape=(5, 5, 1)
      sharding=None
    ] l
    n:f32[5,5,1] = div m k
    o:bf16[5,5,1] = convert_element_type[new_dtype=bfloat16 weak_type=False] n
    p:bool[] = gt k 0.0:f32[]
    q:bf16[5,5,1] r:bf16[5,5,1] = jit[
      name=_where
      jaxpr={ lambda ; p:bool[] o:bf16[5,5,1] s:f32[].

In [10]:
model_weights = updated_weights[:-1]
embedding_table = updated_weights[-1]
input_embeddings = embedding_table[tokens[:, :-1]]

logits, pullback = jax.vjp(
    lambda embeddings: forward_logits(embeddings, model_weights),
    input_embeddings,
)

output_seed = jnp.zeros_like(logits).at[0, 2, 42].set(1)
(input_grads,) = pullback(output_seed)

print(input_grads[0, :, :])

[[0.000225067 0.000225067 0.000225067 0.000225067 0.000225067 0.000225067
  0.000225067 0.000225067]
 [-1.14844 -1.14844 -1.14844 -1.14844 -1.14844 -1.14844 -1.14844 -1.14844]
 [0.554688 -0.169922 -1.26562 -2.10938 0.992188 -1.4375 1.76562 1.29688]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]


In [11]:
def find_tangent(direction):
  _, output_tangents = jax.jvp(lambda embeddings:
                              forward_logits(embeddings, updated_weights[:-1]),
                                (input_embeddings,),
                                (direction,)
                              )
  return output_tangents

direction = jnp.zeros_like(input_embeddings).at[0, 2, 0].set(1)
print(find_tangent(direction)[0, 2, 42])
print(input_grads[0, 2, 0])

0.507812
0.554688


In [12]:
directions = jnp.stack([jnp.zeros_like(input_embeddings).at[0, i, 0].set(1) for i in range(3)])

tangents = jax.vmap(find_tangent)(directions)
print(tangents[0, :, :, :])

[[[0.371094 -0.546875 0.498047 ... 0.451172 0.257812 0.964844]
  [-2.58684e-05 -5.10216e-05 -1.96695e-05 ... 2.74181e-05 9.63211e-05
   2.25306e-05]
  [-0.000211716 -8.91685e-05 4.95911e-05 ... 0.000207901 0.000185966
   -4.31538e-05]
  [1.39062 0.621094 1.46094 ... 0.875 0.427734 0.820312]
  [-2.41474e-15 3.45557e-15 -8.74301e-16 ... -1.17267e-15 1.58901e-15
   -1.97065e-15]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]]


In [13]:
full_directions = jnp.stack([
        jnp.zeros_like(input_embeddings).at[i, j, k].set(1)
        for i in range(input_embeddings.shape[0])
        for j in range(input_embeddings.shape[1])
        for k in range(input_embeddings.shape[2])
])

full_tangents = jax.vmap(find_tangent)(full_directions)
print(full_tangents)

full_tg_jacfwd = jax.jacfwd(forward_logits, argnums=0)
print(full_tg_jacfwd(input_embeddings, updated_weights[:-1]))

Streaming output truncated to the last 5000 lines.
   [0 0 0 ... 0 0 0]]

  [[0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]]

  [[0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]]

  [[0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]]

  [[0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [0 0 0 ... 0 0 0]
   [-0.287109 0.933594 0.0742188 ... -0.15918 -0.170898 -0.445312]]]]
[[[[[[0.371094 -0.245117 0.445312 ... 0.120605 -0.00860596 -0.554688]
     [0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]]

    [[0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]]

    [[0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]]

    [[0 0 0 ... 0 0 0]
     [0 0 0 ... 0 0 0]
     [0

In [14]:
def step_wrapper(input_weights, tokens, l_rate):
  def step_body(carry, _):
    weights = carry
    new_weights, loss = training_step(tokens, weights, l_rate)

    return new_weights, loss

  final_weights, losses = jax.lax.scan(
      step_body,
      init=input_weights,
      length=100
  )
  return final_weights, losses[-1]

final_weights, loss = step_wrapper(weights, tokens, l_rate)
print(loss, decoder_layer(tokens, final_weights))

11.625 11.625


In [15]:
print("=============================== AFTER SCAN JIT ===============================")
print(jax.jit(step_wrapper).trace(weights, tokens, l_rate).jaxpr)

print("=============================== AFTER FOR  JIT ===============================")
print(jax.jit(run_epochs, static_argnames=("epoch_count",)).trace(tokens, weights, l_rate, 5).jaxpr)



=============================== AFTER SCAN JIT ===============================
let _var = { lambda ; a:bf16[5,5,8] b:i32[]. let
    c:f32[5,5,8] = convert_element_type[new_dtype=float32 weak_type=False] a
    d:f32[5,5] = reduce_sum[axes=(2,)] c
    e:f32[5,5,1] = broadcast_in_dim[
      broadcast_dimensions=(0, 1)
      shape=(5, 5, 1)
      sharding=None
    ] d
    f:f32[5,5,1] = div e 8.0:f32[]
    g:f32[5,5,8] = sub c f
    h:f32[5,5,8] = square g
    i:f32[5,5,8] = mul 2.0:f32[] g
    j:f32[] = convert_element_type[new_dtype=float32 weak_type=False] b
    k:f32[] = sub 8.0:f32[] j
    l:f32[5,5] = reduce_sum[axes=(2,)] h
    m:f32[5,5,1] = broadcast_in_dim[
      broadcast_dimensions=(0, 1)
      shape=(5, 5, 1)
      sharding=None
    ] l
    n:f32[5,5,1] = div m k
    o:bf16[5,5,1] = convert_element_type[new_dtype=bfloat16 weak_type=False] n
    p:bool[] = gt k 0.0:f32[]
    q:bf16[5,5,1] r:bf16[5,5,1] = jit[
      name=_where
      jaxpr={ lambda ; p:bool[] o:bf16[5,5,1] s:f32